# 第 5 章 §5.3 · Tool Use

一句话：**模型不会执行工具，它只会"请求"调用工具**——真正去查的是你的代码，`AgentExecutor` 只是把「问模型 / 执行工具」这个循环包起来了。

书里原 notebook 有**两段代码粘在同一个 cell 里**（`gemini-pro` 那段和 `gemini-2.0-flash` 那段，作者迭代痕迹），这里只保留一份，逻辑一致。

kernel 选 `Python (agentic-book)`。执行时序、七个关键点（docstring 如何决定调用、tool_call_id、异常降级）、生产清单都在 `../../ch05_tool_use/README.md`。

In [1]:
import asyncio
import sys
import threading
from pathlib import Path

# 【适配】向上找含 common/ 的目录
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

from common import make_llm

# 【适配】★ LangChain 1.x 把 v0.3 的 agent 体系整体搬到了 langchain_classic，
#   从 langchain.agents 导入会 ImportError。
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

# 必须用支持工具调用的模型。纯文本补全模型根本没有 tool_calls 这个概念。
llm = make_llm(temperature=0)


# ★ 本章新增 @tool：做三件事 —— name ← 函数名；description ← docstring
#   （模型靠它决定要不要调）；args_schema ← 类型注解 + Args 段 → JSON Schema。
# ⚠️ docstring 写成"搜索信息"这种含糊描述，模型会拒绝调用、改用自己的知识回答。
#    要写清楚「什么时候该用它」，而不只是「它做什么」。
@tool
def search_information(query: str) -> str:
    """Provides factual information on a given topic. Use this tool to find answers to questions
    like 'What is the capital of France?' or 'What is the weather in London?'.
    """
    print(f"\n--- 🛠️ Tool Called: search_information with query: '{query}' ---")

    simulated_results = {
        "weather in london": "The weather in London is currently cloudy with a temperature of 15°C.",
        "capital of france": "The capital of France is Paris.",
        "population of earth": "The estimated population of Earth is around 8 billion people.",
        "tallest mountain": "Mount Everest is the tallest mountain above sea level.",
    }

    # ⚠️ 匹配策略很关键：模型传的往往是【完整问句】"What's the weather like in London?",
    #   精确匹配、子串匹配都会失手（多一个 "like" 就挂）。词元全包含最稳。
    q_tokens = set(query.lower().replace("?", " ").replace("'", " ").split())
    result = next(
        (v for k, v in simulated_results.items() if set(k.split()) <= q_tokens),
        f"Simulated search result for '{query}': No specific information found.",
    )

    print(f"--- TOOL RESULT: {result} ---")
    return result


tools = [search_information]

# ★ 那个 ("placeholder", "{agent_scratchpad}") 不是可选项：它是 AgentExecutor 回填
#   中间过程的槽位。少了它，模型看不到自己上一轮做过什么，典型症状是反复调同一个工具
#   直到撞上限。
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)

# AgentExecutor = while 循环 + 工具注册表 + 迭代上限。
# max_iterations 是必需的保险丝，防止模型死循环烧 token；
# verbose=True 会打印完整轨迹，学习阶段强烈建议打开。
agent_executor = AgentExecutor(
    agent=agent,
    verbose=True,
    tools=tools,
    max_iterations=5,
)


async def run_agent_with_tool(query: str) -> None:
    print(f"\n--- 🏃 Running Agent with Query: '{query}' ---")
    try:
        response = await agent_executor.ainvoke({"input": query})
        print("\n--- ✅ Final Agent Response ---")
        print(response["output"])
    except Exception as e:                       # noqa: BLE001
        print(f"\n🛑 An error occurred during agent execution: {e}")


async def main() -> None:
    """书里把三个查询并发跑 —— 这是第 3 章并行模式的实际应用。"""
    await asyncio.gather(
        run_agent_with_tool("What is the capital of France?"),
        run_agent_with_tool("What's the weather like in London?"),
        run_agent_with_tool("Tell me something about dogs."),
    )


def run_in_own_loop(async_fn):
    """【适配】notebook 里事件循环已在运行，asyncio.run 会报 cannot be called from a
    running event loop，所以用独立线程 + 独立事件循环（同 Ch3）。"""
    box = {}

    def _target():
        try:
            box["result"] = asyncio.run(async_fn())
        except BaseException as e:               # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


if __name__ == "__main__":
    run_in_own_loop(main)


--- 🏃 Running Agent with Query: 'What is the capital of France?' ---

--- 🏃 Running Agent with Query: 'What's the weather like in London?' ---

--- 🏃 Running Agent with Query: 'Tell me something about dogs.' ---


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...



Invoking: `search_information` with `{'query': 'What is the capital of France?'}`



--- 🛠️ Tool Called: search_information with query: 'What is the capital of France?' ---
--- TOOL RESULT: The capital of France is Paris. ---
The capital of France is Paris.
Invoking: `search_information` with `{'query': "What's the weather like in London?"}`



--- 🛠️ Tool Called: search_information with query: 'What's the weather like in London?' ---
--- TOOL RESULT: The weather in London is currently cloudy with a temperature of 15°C. ---
The weather in London is currently cloudy with a temperature of 15°C.


Invoking: `search_information` with `{'query': 'facts about dogs'}`
responded: I'll look up some information about dogs for you.


--- 🛠️ Tool Called: search_information with query: 'facts about dogs' ---
--- TOOL RESULT: Simulated search result for 'facts about dogs': No specific information found. ---
Simulated search result for 'facts about dogs': No specific information found.

The capital of France is **Paris**.

> Finished chain.

--- ✅ Final Agent Response ---
The capital of France is **Paris**.


The weather in London is currently **cloudy** with a temperature of **15°C**. Let me know if you'd like more details!

> Finished chain.

--- ✅ Final Agent Response ---
The weather in London is currently **cloudy** with a temperature of **15°C**. Let me know if you'd like more details!

Invoking: `search_information` with `{'query': 'what are dogs, history and characteristics of domesticated dogs'}`
responded: Let me try a more specific search.


--- 🛠️ Tool Called: search_information with query: 'what are dogs, history and characteristics of domesticated dogs' ---
--- TOOL RESULT: Simulated search result for 'what are dogs, history and characteristics of domesticated dogs': No specific information found. ---
Simulated search result for 'what are dogs, history and characteristics of domesticated dogs': No specific information found.

The search tool isn't returning any results right now, but I'm happy to share some general knowledge about dogs:

**Dogs** (*Canis lupus familiaris*) are domesticated mammals and one of the most popular companion animals in the world. Here are some interesting things about them:

- **Ancient companions:** Dogs were the first animals to be domesticated, likely over 15,000 years ago. They descended from wolves and have lived alongside humans for millennia as hunting partners, guardians, and companions.

- **Incredible senses:** Dogs have a sense of smell that's roughly 10,000 to 100,000 times more acute than humans'. They also hear higher-pitched sounds that we can't detect.

- **Amazing diversity:** From tiny Chihuahuas to giant Great Danes, dogs come in an extraordinary range of sizes, shapes, and coats—more physical variety than almost any other mammal species.

- **Intelligence and emotion:** Dogs can learn hundreds of words and commands, and they're known to experience a range of em